In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
from getDbdata import get_raw_scrapes
from saveLLMText import save_llm_text

RuntimeError: SUPABASE_URL and SUPABASE_KEY must be set in the environment

In [4]:
# Correct way
llm_input = get_raw_scrapes(limit=50)  # ✅ call the function

our_data = []
competitor_data = []

for record in llm_input:
    if record.get("is_our_site"):
        our_data.append(record["raw_data"])
    else:
        competitor_data.append(record["raw_data"])

print(f"Our site data count: {len(our_data)}")
print(f"Competitor data count: {len(competitor_data)}")


Our site data count: 1
Competitor data count: 0


In [ ]:
os.environ["GROQ_API_KEY"] = "REMOVED_SECRET"

In [ ]:
%pip install groq

In [7]:
from groq import Groq

In [18]:
client = Groq() 

In [11]:
import json
def chunk_json(data, max_chars=6000):
    text = json.dumps(data)
    return [text[i:i+max_chars] for i in range(0, len(text), max_chars)]


In [12]:
competitor_chunks = chunk_json(competitor_data)
our_chunks = chunk_json(our_data)


In [13]:
COMPETITOR_SYSTEM_PROMPT = """
You are a senior competitive intelligence analyst and product strategist.

Your task is to extract HIGH-SIGNAL, DEFENSIBLE insights from competitor data.

STRICT RULES:
- Use ONLY provided evidence.
- No speculation about revenue, traffic, growth, market share.
- No repetition of raw data.

ANALYTICAL STANDARD:
Separate clearly:
(1) Observed Signals
(2) Inferred Strategy (with justification)
(3) Actionable Implications
"""


In [19]:
competitor_results = []

for i, chunk in enumerate(competitor_chunks):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": COMPETITOR_SYSTEM_PROMPT},
            {"role": "user", "content": f"""
This is CHUNK {i+1} of competitor data.

DATA:
{chunk}

Return ONLY:

### Observed Signals
### Inferred Strategy (with reasoning)
### Actionable Implications
"""}
        ],
        max_tokens=1000,
        temperature=0.2
    )

    competitor_results.append(response.choices[0].message.content)

competitor_summary = "\n\n".join(competitor_results)
save_llm_text("competitorSummary", competitor_summary)

✅ Saved competitorSummary


In [20]:
competitor_summary

'### Observed Signals\n\n- **Product Diversification**: Competitor offers multiple products/services, including [list of products/services].\n- **Target Audience Segmentation**: Competitor has a distinct approach to targeting different audience segments, as evident from their [list of features or content tailored to specific segments].\n- **Partnerships and Collaborations**: Competitor has formed partnerships with [list of partners], indicating a focus on expanding their reach and offerings.\n\n### Inferred Strategy\n\n- **Diversification Strategy**: Competitor is employing a diversification strategy to reduce dependence on a single product or service, increasing their overall resilience and adaptability in the market.\n- **Segmentation and Targeting**: Competitor is using a segmentation and targeting strategy to effectively cater to diverse customer needs, increasing customer satisfaction and loyalty.\n- **Expansion through Partnerships**: Competitor is leveraging partnerships to expa

In [21]:
OUR_SYSTEM_PROMPT = """
You are a senior product and growth analyst analyzing OUR company.

Your role is to identify:
- Strengths
- Weaknesses
- Gaps
- Underutilized advantages

Use ONLY evidence in the data.
No aspirational claims.
"""


In [22]:
our_results = []

for i, chunk in enumerate(our_chunks):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": OUR_SYSTEM_PROMPT},
            {"role": "user", "content": f"""
This is CHUNK {i+1} of our company data.

DATA:
{chunk}

Return ONLY:

### Observed Signals
### Internal Strengths & Weaknesses
### Improvement Opportunities
"""}
        ],
        max_tokens=1000,
        temperature=0.2
    )

    our_results.append(response.choices[0].message.content)

our_summary = "\n\n".join(our_results)
print(our_summary)
save_llm_text("ourSummary", our_summary)

### Observed Signals

Based on the provided data, the following signals are observed:

- **Highly Engaging Features**: The website features a prominent call-to-action (CTA) "Book a demo" on the homepage, indicating a strong focus on conversion.
- **Strong Social Media Presence**: The company has a presence on multiple social media platforms, including Twitter, YouTube, Facebook, LinkedIn, and Instagram.
- **Robust Integration Capabilities**: The website highlights seamless integrations with various tools and platforms, suggesting a focus on scalability and compatibility.
- **Positive Testimonials**: The website features customer testimonials, showcasing the company's ability to deliver value to its clients.
- **Limited Pricing Information**: The pricing section is empty, indicating a potential gap in transparency and clarity around pricing models.

### Internal Strengths & Weaknesses

Based on the observed signals, the following internal strengths and weaknesses are identified:

**Stre

In [23]:
def chunk_text(text, max_chars=3500):
    return [text[i:i+max_chars] for i in range(0, len(text), max_chars)]


In [24]:
our_summary_chunks = chunk_text(our_summary)
competitor_summary_chunks = chunk_text(competitor_summary)


In [25]:
COMPARISON_SYSTEM_PROMPT = """
You are a principal competitive intelligence analyst advising a founder.

Your role is to perform EVIDENCE-BASED COMPARATIVE ANALYSIS between:
- Our company
- One or two competitor companies

STRICT RULES:
- Base conclusions ONLY on provided data.
- Do NOT speculate about revenue, traffic, growth, funding, or market share.
- Do NOT reward competitors for polish unless it implies strategic intent.
- Do NOT invent missing features or capabilities.

ANALYTICAL STANDARDS:
- Think in relative advantage, not absolute description.
- Identify leverage points, not feature checklists.
- Distinguish clearly between:
  (1) Observed Signals
  (2) Relative Strengths / Weaknesses
  (3) Strategic Implications
  (4) Offensive Recommendations

PRIMARY OBJECTIVE:
Help our company understand how to OUTMANEUVER competitors,
not merely match them.
"""


In [26]:
comparison_chunks = []

for i in range(min(len(our_summary_chunks), len(competitor_summary_chunks))):
    comparison_chunks.append({
        "our": our_summary_chunks[i],
        "competitor": competitor_summary_chunks[i]
    })


In [27]:
comparison_results = []

for i, pair in enumerate(comparison_chunks):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": COMPARISON_SYSTEM_PROMPT},
            {"role": "user", "content": f"""
COMPARISON CHUNK {i+1}

## OUR COMPANY SIGNALS
{pair['our']}

## COMPETITOR SIGNALS
{pair['competitor']}

TASK:
Extract ONLY NEW, non-redundant comparative insights.

OUTPUT FORMAT:

### Relative Strengths (Us vs Them)
### Relative Weaknesses (Us vs Them)
### Strategic Gaps & Leverage Points
### Where We Can Win Decisively
### What Not To Compete On
"""}
        ],
        max_tokens=900,
        temperature=0.2
    )

    comparison_results.append(response.choices[0].message.content)
comparison_report = "\n\n".join(comparison_results)
save_llm_text("comparisonReport", comparison_report)

comparison_report

✅ Saved comparisonReport


"### Relative Strengths (Us vs Them)\n\n- **Conversion-Focused Design**: Our company's prominent CTA on the homepage is a relative strength compared to the competitor's lack of observed signals related to conversion-focused design.\n- **Scalable Integration Capabilities**: Our company's emphasis on seamless integrations with various tools and platforms is a relative strength compared to the competitor's lack of observed signals related to integration capabilities.\n- **Positive Customer Experience**: Our company's customer testimonials suggest a strong focus on delivering value to clients, which is a relative strength compared to the competitor's lack of observed signals related to customer experience.\n\n### Relative Weaknesses (Us vs Them)\n\n- **Limited Pricing Transparency**: Our company's empty pricing section is a relative weakness compared to the competitor's lack of observed signals related to pricing transparency.\n- **Limited Social Media Engagement**: Our company's zero foll

In [28]:
strategy_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": """
You are a founder-level strategic advisor.

Your job is to turn competitive intelligence
into clear, prioritized business actions.
"""
        },
        {
            "role": "user",
            "content": f"""
INPUT:
{comparison_report}

TASK:
Produce the BEST COURSE OF ACTION.

OUTPUT FORMAT:

## Strategic Diagnosis
## Immediate Actions (0–30 days)
## Medium-Term Plays (30–90 days)
## Long-Term Advantage
## Risks to Monitor
"""
        }
    ],
    max_tokens=1200,
    temperature=0.2
)

final_strategy = strategy_response.choices[0].message.content
save_llm_text("finalStategy", final_strategy)
print(final_strategy)


✅ Saved finalStategy
## Strategic Diagnosis

Based on the competitive intelligence analysis, our company has relative strengths in conversion-focused design, scalable integration capabilities, and positive customer experience. However, we have relative weaknesses in limited pricing transparency and limited social media engagement. The competitor's lack of observed signals related to pricing transparency and increasing engagement on social media creates strategic gaps for our company to develop a clear and transparent pricing model and enhance social media engagement.

## Immediate Actions (0–30 days)

1. **Develop a Clear Pricing Model**: Create a transparent pricing page that outlines our pricing strategy, features, and benefits. This will help establish trust with potential customers and differentiate us from the competitor.
2. **Enhance Social Media Presence**: Allocate resources to create social media accounts on Twitter and LinkedIn, and start engaging with our target audience by 

In [ ]:
# Run this in a separate terminal to auto-trigger analysis:
# cd LLM
# python auto_trigger.py

# Or manually trigger with:
print("✅ To enable auto-trigger: python auto_trigger.py")
print("📊 Current setup will:")
print("   - Monitor database every 60 seconds")
print("   - Run analysis when 5+ new records appear")
print("   - Save results automatically")

## Auto-Trigger Monitor

To automatically run this analysis when new scraper data arrives, use the trigger script in a separate terminal.